# MiseCast Exploration Notebook
#### Team SVNAD

A notebook file to examine components such as data and sanity checks.

---

In [1]:
# Every runtime restart, this line needs to be run first
# Uncomment the code below if using Google Colab's runtime
# %pip install -r requirements.txt

In [2]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import json

sys.path.append(str(Path.cwd().parent))

from sklearn.ensemble import HistGradientBoostingRegressor
from utils import mongo_export as mx

from utils import data_handling as dh

In [3]:
DATA_DIR = Path.cwd().parent / "data"

menu_items = dh.load_menu_items(DATA_DIR / "menu_items.csv")
menu_variants = dh.load_menu_variants(DATA_DIR / "menu_variants.csv")
menu_recipes = dh.load_menu_recipes(DATA_DIR / "menu_recipes.csv")
ingredient_master = dh.load_ingredient_master(DATA_DIR / "ingredient_master.csv")
sales = dh.load_historical_sales(DATA_DIR / "historical_sales.csv")

dh.check_joins(menu_variants, menu_recipes, ingredient_master, sales)

All joins check out: recipes -> variants -> sales, and recipes -> ingredient_master.


In [4]:
def load_calendar_context(path):
    df = pd.read_csv(path, sep=";")
    df.columns = [c.strip() for c in df.columns]
    df["date"] = pd.to_datetime(df["date"])
    return df


def load_reservations(path):
    df = pd.read_csv(path, sep=";")
    df.columns = [c.strip() for c in df.columns]
    df["date"] = pd.to_datetime(df["date"])
    return df


def load_weather(path):
    df = pd.read_csv(path, sep=";")
    df.columns = [c.strip() for c in df.columns]
    df["date"] = pd.to_datetime(df["date"])
    return df


def load_local_events(path):
    df = pd.read_csv(path, sep=";")
    df.columns = [c.strip() for c in df.columns]
    df["event_date"] = pd.to_datetime(df["event_date"])
    return df


def load_promotion_history(path):
    df = pd.read_csv(path, sep=";")
    df.columns = [c.strip() for c in df.columns]
    df["promotion_date"] = pd.to_datetime(df["promotion_date"])
    return df


def load_current_inventory(path):
    df = pd.read_csv(path, sep=";")
    df.columns = [c.strip() for c in df.columns]
    df["inventory_as_of_date"] = pd.to_datetime(df["inventory_as_of_date"])
    return df


def load_inventory_batches_expiry(path):
    df = pd.read_csv(path, sep=";")
    df.columns = [c.strip() for c in df.columns]
    df["received_date"] = pd.to_datetime(df["received_date"])
    df["expiry_date"] = pd.to_datetime(df["expiry_date"])
    return df

def load_suppliers(path):
    df = pd.read_csv(path, sep=";")
    df.columns = [c.strip() for c in df.columns]
    return df

In [5]:
calendar = load_calendar_context(DATA_DIR / "calendar_context.csv")
reservations = load_reservations(DATA_DIR / "reservations.csv")
weather = load_weather(DATA_DIR / "weather.csv")
events = load_local_events(DATA_DIR / "local_events.csv")
promotions = load_promotion_history(DATA_DIR / "promotion_history.csv")
current_inventory = load_current_inventory(DATA_DIR / "current_inventory.csv")
batches = load_inventory_batches_expiry(DATA_DIR / "inventory_batches_expiry.csv")
suppliers = load_suppliers(DATA_DIR / "suppliers.csv")

### 1. Feature Engineering

In [6]:
def build_feature_table(sales, menu_variants, calendar, reservations, weather, events, promotions):
    # aggregate raw sales rows (per channel) up to one row per date/service/variant
    agg = (sales.groupby(["date", "service_period", "menu_variant_id", "menu_item_id"], as_index=False)
                 .agg(quantity_sold=("quantity_sold", "sum")))

    # drop variants the data itself flags as not trainable (e.g. the inactive MI-067)
    eligible = set(menu_variants.loc[menu_variants.demand_model_training_eligible, "menu_variant_id"])
    df = agg[agg.menu_variant_id.isin(eligible)].reset_index(drop=True)

    cal_cols = ["date", "day_of_week", "month", "season", "is_weekend", "is_public_holiday",
                "is_school_holiday", "special_occasion_demand_factor"]
    df = df.merge(calendar[cal_cols], on="date", how="left")

    # only "actual" rows for training -- "forecast" rows are for future prediction, not history
    res_actual = reservations[reservations.record_type == "actual"][["date", "service_period", "reserved_covers"]]
    df = df.merge(res_actual, on=["date", "service_period"], how="left")
    df["reserved_covers"] = df["reserved_covers"].fillna(df["reserved_covers"].median())

    wx_actual = weather[weather.record_type == "actual"][
        ["date", "average_temperature_c", "rain_probability_pct", "is_hot_day", "is_cold_day"]]
    df = df.merge(wx_actual, on="date", how="left")

    df = df.merge(menu_variants[["menu_variant_id", "category", "portion_size"]], on="menu_variant_id", how="left")

    def event_score(row):
        relevant = events[events.service_period_impacted.isin([row.service_period, "Both"])]
        if relevant.empty:
            return 0.0
        day_diff = (relevant.event_date - row.date).abs().dt.days
        weight = relevant.demand_relevance_score / (1 + relevant.distance_from_restaurant_km)
        if (day_diff <= 0).any():
            return float(weight[day_diff <= 0].max())
        nearest = day_diff.idxmin()
        return float(weight.loc[nearest] * np.exp(-day_diff.loc[nearest] / 2.0))

    df["event_proximity"] = df.apply(event_score, axis=1)

    promo_keys = set(zip(promotions.promotion_date, promotions.service_period, promotions.menu_variant_id))
    df["promotion_active"] = df.apply(
        lambda r: (r.date, r.service_period, r.menu_variant_id) in promo_keys, axis=1)

    # matching-weekday lag/rolling (per variant+service_period+weekday) -- this IS the baseline too
    df = df.sort_values(["menu_variant_id", "service_period", "day_of_week", "date"]).reset_index(drop=True)
    g_weekday = df.groupby(["menu_variant_id", "service_period", "day_of_week"])["quantity_sold"]
    df["lag_7d"] = g_weekday.shift(1)
    df["rolling_matching_weekday_avg"] = g_weekday.transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean())

    # short-term momentum across consecutive dates, same variant+service_period
    df = df.sort_values(["menu_variant_id", "service_period", "date"]).reset_index(drop=True)
    g_series = df.groupby(["menu_variant_id", "service_period"])["quantity_sold"]
    df["recent_momentum"] = g_series.transform(lambda s: s.diff().rolling(3, min_periods=1).mean().shift(1))

    overall_mean = df.groupby("menu_variant_id")["quantity_sold"].transform("mean")
    df["lag_7d"] = df["lag_7d"].fillna(overall_mean)
    df["rolling_matching_weekday_avg"] = df["rolling_matching_weekday_avg"].fillna(overall_mean)
    df["recent_momentum"] = df["recent_momentum"].fillna(0.0)

    return df

In [7]:
hist = build_feature_table(sales, menu_variants, calendar, reservations, weather, events, promotions)

### 2. The Pooled Model

In [8]:
CATEGORICAL = ["menu_variant_id", "menu_item_id", "category", "portion_size",
               "service_period", "day_of_week", "month", "season"]
NUMERIC = ["is_weekend", "is_public_holiday", "is_school_holiday", "special_occasion_demand_factor",
           "reserved_covers", "average_temperature_c", "rain_probability_pct", "is_hot_day",
           "is_cold_day", "event_proximity", "promotion_active", "lag_7d",
           "rolling_matching_weekday_avg", "recent_momentum"]
FEATURES = CATEGORICAL + NUMERIC


def train_quantile_models(train_df, target="quantity_sold"):
    df = train_df.copy()
    for c in CATEGORICAL:
        df[c] = df[c].astype("category")
    for c in NUMERIC:
        if df[c].dtype == bool:
            df[c] = df[c].astype(int)

    models = {}
    for name, q in {"p50": 0.5, "p90": 0.9}.items():
        m = HistGradientBoostingRegressor(
            loss="quantile", quantile=q, max_depth=6, max_iter=300,
            min_samples_leaf=15, learning_rate=0.06, random_state=42,
            categorical_features="from_dtype",
        )
        m.fit(df[FEATURES], df[target])
        models[name] = m
    return models


def predict(models, feature_df):
    out = feature_df.copy()
    for c in CATEGORICAL:
        out[c] = out[c].astype("category")
    for c in NUMERIC:
        if out[c].dtype == bool:
            out[c] = out[c].astype(int)
    out["p50"] = models["p50"].predict(out[FEATURES])
    out["p90"] = np.maximum(models["p90"].predict(out[FEATURES]), out["p50"])
    return out

### 3. Evaluation vs Required Baseline

In [9]:
def wape(actual, forecast):
    actual, forecast = np.asarray(actual, float), np.asarray(forecast, float)
    return np.abs(actual - forecast).sum() / actual.sum()


def evaluate(holdout_with_preds, target="quantity_sold"):
    holdout_with_preds["baseline"] = holdout_with_preds["rolling_matching_weekday_avg"]
    overall = {
        "model_wape": wape(holdout_with_preds[target], holdout_with_preds["p50"]),
        "baseline_wape": wape(holdout_with_preds[target], holdout_with_preds["baseline"]),
    }
    by_category = (holdout_with_preds.groupby("category", observed=True)
                   .apply(lambda g: pd.Series({
                       "model_wape": wape(g[target], g["p50"]),
                       "baseline_wape": wape(g[target], g["baseline"]),
                   })))
    by_category["beats_baseline"] = by_category.model_wape < by_category.baseline_wape
    return overall, by_category

### 4. Future-Window Forecast

In [10]:
def build_future_feature_table(hist, menu_variants, calendar, reservations, weather, events, promotions):
    last_actual_date = hist["date"].max()
    future_dates = calendar.loc[calendar["date"] > last_actual_date, "date"]
    if future_dates.empty:
        raise ValueError("No future dates found beyond the last actual sales date.")

    eligible = menu_variants.loc[menu_variants.demand_model_training_eligible,
                                  ["menu_variant_id", "menu_item_id", "category", "portion_size"]]
    service_periods = sorted(hist["service_period"].unique())

    skeleton = (
        eligible.assign(key=1)
        .merge(pd.DataFrame({"service_period": service_periods, "key": 1}), on="key")
        .merge(pd.DataFrame({"date": future_dates, "key": 1}), on="key")
        .drop(columns="key")
    )

    cal_cols = ["date", "day_of_week", "month", "season", "is_weekend", "is_public_holiday",
                "is_school_holiday", "special_occasion_demand_factor"]
    df = skeleton.merge(calendar[cal_cols], on="date", how="left")

    res_fcst = reservations[reservations.record_type == "forecast"][["date", "service_period", "reserved_covers"]]
    df = df.merge(res_fcst, on=["date", "service_period"], how="left")
    df["reserved_covers"] = df["reserved_covers"].fillna(hist["reserved_covers"].median())

    wx_fcst = weather[weather.record_type == "forecast"][
        ["date", "average_temperature_c", "rain_probability_pct", "is_hot_day", "is_cold_day"]]
    df = df.merge(wx_fcst, on="date", how="left")
    for c in ["average_temperature_c", "rain_probability_pct"]:
        df[c] = df[c].fillna(hist[c].median())
    for c in ["is_hot_day", "is_cold_day"]:
        df[c] = df[c].fillna(False)

    def event_score(row):
        relevant = events[events.service_period_impacted.isin([row.service_period, "Both"])]
        if relevant.empty:
            return 0.0
        day_diff = (relevant.event_date - row.date).abs().dt.days
        weight = relevant.demand_relevance_score / (1 + relevant.distance_from_restaurant_km)
        if (day_diff <= 0).any():
            return float(weight[day_diff <= 0].max())
        nearest = day_diff.idxmin()
        return float(weight.loc[nearest] * np.exp(-day_diff.loc[nearest] / 2.0))

    df["event_proximity"] = df.apply(event_score, axis=1)

    promo_keys = set(zip(promotions.promotion_date, promotions.service_period, promotions.menu_variant_id))
    df["promotion_active"] = df.apply(
        lambda r: (r.date, r.service_period, r.menu_variant_id) in promo_keys, axis=1)

    tail_stats = (
        hist.sort_values("date")
            .groupby(["menu_variant_id", "service_period", "day_of_week"])
            .agg(lag_7d=("quantity_sold", "last"),
                 rolling_matching_weekday_avg=("quantity_sold", lambda s: s.tail(4).mean()))
            .reset_index()
    )
    momentum_last = (
        hist.sort_values("date")
            .groupby(["menu_variant_id", "service_period"])["recent_momentum"]
            .last()
            .reset_index()
    )
    df = df.merge(tail_stats, on=["menu_variant_id", "service_period", "day_of_week"], how="left")
    df = df.merge(momentum_last, on=["menu_variant_id", "service_period"], how="left")

    overall_mean = hist.groupby("menu_variant_id")["quantity_sold"].mean()
    fallback = df["menu_variant_id"].map(overall_mean)
    df["lag_7d"] = df["lag_7d"].fillna(fallback)
    df["rolling_matching_weekday_avg"] = df["rolling_matching_weekday_avg"].fillna(fallback)
    df["recent_momentum"] = df["recent_momentum"].fillna(0.0)

    return df

In [11]:
models = train_quantile_models(hist)                                    # hist = build_feature_table(...) output
future = build_future_feature_table(hist, menu_variants, calendar, reservations, weather, events, promotions)
forecast = predict(models, future)

### 5. Expected Revenue

In [12]:
price_lookup = menu_variants.set_index("menu_variant_id")["sale_price_aud"].astype(float)
forecast["expected_revenue_p50"] = forecast["p50"] * forecast["menu_variant_id"].map(price_lookup)
forecast["expected_revenue_p90"] = forecast["p90"] * forecast["menu_variant_id"].map(price_lookup)

daily_revenue = forecast.groupby("date")[["expected_revenue_p50", "expected_revenue_p90"]].sum()

### 6. Run Out Prediction (Section 4.2)

In [13]:
from utils import order_optimization as oo

explode_ingredient_demand = oo.explode_ingredient_demand
usable_stock_by_date = oo.usable_stock_by_date

def explode_ingredient_demand(forecast, menu_recipes, quantity_col="p50"):
    eligible = menu_recipes[menu_recipes["inventory_consumption_eligible"].astype(str).str.lower() == "true"]
    merged = forecast[["date", "service_period", "menu_variant_id", quantity_col]].merge(
        eligible[["menu_variant_id", "ingredient_id", "ingredient_name", "effective_quantity_per_portion", "unit"]],
        on="menu_variant_id", how="inner",
    )
    merged["ingredient_demand"] = merged[quantity_col] * merged["effective_quantity_per_portion"]
    return (merged.groupby(["date", "ingredient_id", "ingredient_name", "unit"], as_index=False)
                  .agg(ingredient_demand=("ingredient_demand", "sum")))


def usable_stock_by_date(batches, dates):
    """Usable stock per ingredient on a given date = sum of batches whose
    expiry_date hasn't passed yet. A batch that's already expired by that
    date can't be used -- it's waste, not stock (that's Section 4.4's concern,
    but it still has to be excluded here)."""
    dates = pd.Series(sorted(pd.to_datetime(dates).unique()))
    rows = []
    for d in dates:
        usable = batches[batches["expiry_date"] >= d]
        rows.append(usable.groupby("ingredient_id")["quantity_remaining"].sum().rename(d))
    return pd.concat(rows, axis=1)


def predict_runout(forecast, menu_recipes, batches, quantity_col="p50"):
    daily_demand = explode_ingredient_demand(forecast, menu_recipes, quantity_col)
    dates = sorted(daily_demand["date"].unique())
    stock_by_date = usable_stock_by_date(batches, dates)

    demand_pivot = (daily_demand.pivot(index="ingredient_id", columns="date", values="ingredient_demand")
                                 .fillna(0.0).reindex(columns=dates, fill_value=0.0))

    all_ingredients = sorted(set(demand_pivot.index) | set(stock_by_date.index))
    demand_pivot = demand_pivot.reindex(all_ingredients, fill_value=0.0)
    stock_by_date = stock_by_date.reindex(all_ingredients)

    results = []
    for ing in all_ingredients:
        remaining = None
        run_out_date = None
        for d in dates:
            usable_today = stock_by_date.loc[ing, d]
            usable_today = 0.0 if pd.isna(usable_today) else usable_today
            remaining = usable_today if remaining is None else min(remaining, usable_today)
            remaining -= demand_pivot.loc[ing, d]
            if remaining < 0 and run_out_date is None:
                run_out_date = d
        results.append({
            "ingredient_id": ing,
            "run_out_date": run_out_date,
            "days_until_run_out": None if run_out_date is None else (run_out_date - dates[0]).days,
        })
    return pd.DataFrame(results)


runout_p50 = predict_runout(forecast, menu_recipes, batches, "p50")
runout_p90 = predict_runout(forecast, menu_recipes, batches, "p90")
runout = (runout_p50.merge(runout_p90, on="ingredient_id", suffixes=("_p50", "_p90"))
                     .merge(ingredient_master[["ingredient_id", "ingredient_name", "ingredient_group_code"]],
                            on="ingredient_id", how="left"))

# Menu items that become unservable once a given ingredient runs out
recipe_join = menu_recipes.loc[
    menu_recipes["inventory_consumption_eligible"].astype(str).str.lower() == "true",
    ["ingredient_id", "menu_variant_id", "menu_item_id", "menu_variant_name"]]
affected_items = runout.merge(recipe_join, on="ingredient_id", how="left")

### Getting the Output

In [14]:
forecast_out = forecast[["date", "service_period", "menu_variant_id", "menu_item_id",
                          "p50", "p90", "expected_revenue_p50", "expected_revenue_p90"]].copy()
forecast_out["date"] = forecast_out["date"].dt.strftime("%Y-%m-%d")

# The code below saves the output JSON file under `data/dashboard_data/`
forecast_out.to_json(DATA_DIR / "dashboard_data" / "forecast.json", orient="records", indent=2)

runout_out = runout[["ingredient_id", "ingredient_name", "ingredient_group_code",
                      "run_out_date_p50", "days_until_run_out_p50",
                      "run_out_date_p90", "days_until_run_out_p90"]].copy()
for c in ["run_out_date_p50", "run_out_date_p90"]:
    runout_out[c] = runout_out[c].apply(lambda d: d.strftime("%Y-%m-%d") if pd.notna(d) else None)
runout_out.to_json(DATA_DIR / "dashboard_data" / "runout.json", orient="records", indent=2)

### Section 4.3

In [15]:
orders = oo.build_order_recommendations(forecast, menu_recipes, batches, current_inventory, suppliers)
orders = orders.merge(ingredient_master[["ingredient_id", "ingredient_name"]], on="ingredient_id", how="left")

In [16]:
orders_out = orders[["ingredient_id", "ingredient_name", "order_by_date", "days_until_order_by", "urgency",
                      "recommended_order_quantity", "unit", "order_packs", "estimated_cost_aud",
                      "supplier_name", "lead_time_days"]].copy()
orders_out["order_by_date"] = orders_out["order_by_date"].dt.strftime("%Y-%m-%d")
orders_out.to_json(DATA_DIR / "dashboard_data" / "orders.json", orient="records", indent=2)

In [17]:
def predict_expiry_waste(forecast, menu_recipes, batches, quantity_col="p50"):
    """FIFO batch consumption: within the forecast window, each day's
    ingredient demand is drawn from the earliest-expiring batch first
    ("use first"). Any batch that reaches its expiry date with quantity
    still unconsumed is predicted waste. Batches expiring after the
    forecast window aren't judged either way -- not enough visibility yet.
    """
    daily_demand = oo.explode_ingredient_demand(forecast, menu_recipes, quantity_col)
    dates = sorted(daily_demand["date"].unique())
    demand_pivot = (daily_demand.pivot(index="ingredient_id", columns="date", values="ingredient_demand")
                                 .fillna(0.0).reindex(columns=dates, fill_value=0.0))

    results = []
    for ingredient_id, ing_batches in batches.groupby("ingredient_id"):
        ing_batches = ing_batches.sort_values("expiry_date")  # FIFO / "use first"
        batch_ids = list(ing_batches["batch_id"])
        remaining = ing_batches.set_index("batch_id")["quantity_remaining"].astype(float).to_dict()
        expiry = ing_batches.set_index("batch_id")["expiry_date"].to_dict()
        unit_cost = ing_batches.set_index("batch_id")["unit_cost_aud"].astype(float).to_dict()
        expired = set()

        daily_row = demand_pivot.loc[ingredient_id] if ingredient_id in demand_pivot.index else pd.Series(0.0, index=dates)

        for d in dates:
            for b in batch_ids:
                if b not in expired and expiry[b] < d:
                    expired.add(b)
            demand_today = float(daily_row.get(d, 0.0))
            for b in batch_ids:
                if demand_today <= 0:
                    break
                if b in expired:
                    continue
                take = min(remaining[b], demand_today)
                remaining[b] -= take
                demand_today -= take

        for b in batch_ids:
            if expiry[b] <= dates[-1] and remaining[b] > 1e-9:
                results.append({
                    "batch_id": b,
                    "ingredient_id": ingredient_id,
                    "expiry_date": expiry[b],
                    "quantity_wasted": remaining[b],
                    "estimated_waste_value_aud": round(remaining[b] * unit_cost[b], 2),
                })
                
    return pd.DataFrame(results)

In [18]:
waste = predict_expiry_waste(forecast, menu_recipes, batches, "p50")
waste = waste.merge(ingredient_master[["ingredient_id", "ingredient_name", "ingredient_group_code"]],
                     on="ingredient_id", how="left")

# Dishes that could be promoted to help sell through an at-risk batch
recipe_join = menu_recipes.loc[
    menu_recipes["inventory_consumption_eligible"].astype(str).str.lower() == "true",
    ["ingredient_id", "menu_variant_id", "menu_item_id", "menu_variant_name"]]
promo_candidates = waste.merge(recipe_join, on="ingredient_id", how="left")

In [19]:
waste_out = waste[["batch_id", "ingredient_id", "ingredient_name", "ingredient_group_code",
                    "expiry_date", "quantity_wasted", "estimated_waste_value_aud"]].copy()
waste_out["expiry_date"] = waste_out["expiry_date"].dt.strftime("%Y-%m-%d")
waste_out.to_json(DATA_DIR / "dashboard_data" / "expiry_waste.json", orient="records", indent=2)

In [20]:
forecast_generated_date = hist["date"].max() + pd.Timedelta(days=1)
menu_items = pd.read_csv(DATA_DIR / "menu_items.csv", sep=";")
menu_items.columns = [c.strip() for c in menu_items.columns]

df_forecast = mx.build_demand_forecast_export(forecast, menu_variants, menu_items, forecast_generated_date)
df_runout = mx.build_runout_predictions_export(forecast, menu_recipes, batches, current_inventory, ingredient_master, waste)
df_orders = mx.build_order_recommendations_export(orders, forecast, menu_recipes, batches, current_inventory, suppliers)
df_expiry = mx.build_expiry_menu_actions_export(waste, batches, menu_recipes, forecast_generated_date)

import os
os.makedirs(DATA_DIR / "mongo_import", exist_ok=True)
for name, df in [("demand_forecast", df_forecast), ("runout_predictions", df_runout),
                  ("order_recommendations", df_orders), ("expiry_menu_actions", df_expiry)]:
    df.to_csv(DATA_DIR / "mongo_import" / f"{name}.csv", sep=";", index=False)